In [9]:
# Feature Pipeline: Unwetterwarnung mit Open-Meteo API

In [10]:
# Hopsworks-Projektverbindungsskript
import os
from pathlib import Path

import hopsworks
from dotenv import load_dotenv

# Sicheres Laden der .env-Datei aus dem Projektstamm
project_root = Path.cwd().resolve()
if not (project_root / ".env").exists():
    project_root = project_root.parent

load_dotenv(project_root / ".env")

# Umgebungsvariablen abrufen
api_key = os.getenv("HOPSWORKS_API_KEY")
project_name = os.getenv("HOPSWORKS_PROJECT_NAME")

if not api_key or not project_name:
    raise ValueError(
        "HOPSWORKS_API_KEY und HOPSWORKS_PROJECT_NAME müssen in der .env-Datei gesetzt sein."
    )

# Verbindung zum Hopsworks-Projekt herstellen
project = hopsworks.login(
    api_key_value=api_key,
    project=project_name,
    host="eu-west.cloud.hopsworks.ai",
    port=443,
)

print(f"✅ Erfolgreich verbunden mit Projekt: {project.name}")

2026-09-15 09:19:11,233 INFO: Closing external client and cleaning up certificates.
2026-09-15 09:19:11,235 INFO: Connection closed.
2026-09-15 09:19:11,236 INFO: Initializing external client
2026-09-15 09:19:11,238 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-09-15 09:19:11,757 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/44167
✅ Erfolgreich verbunden mit Projekt: fhnw_p1_weather_forcasts


In [11]:
# Rohdaten via API abrufen
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry

url = "https://api.open-meteo.com/v1/forecast"


def fetch_weather_data(latitude: float, longitude: float, past_days: int = 7, forecast_days: int = 1) -> pd.DataFrame:
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "pressure_msl",
            "surface_pressure",
            "cloud_cover",
            "wind_speed_10m",
        ],
        "past_days": past_days,
        "forecast_days": forecast_days,
        "timezone": "Europe/Berlin",
    }

    response = openmeteo.weather_api(url, params=params)[0]
    hourly = response.Hourly()
    start_ts = hourly.Time()
    num_points = len(hourly.Variables(0).ValuesAsNumpy())
    start_dt = pd.to_datetime(start_ts, unit="s", utc=True).tz_convert("Europe/Berlin")
    dates = pd.date_range(start=start_dt, periods=num_points, freq="h")

    df = pd.DataFrame({
        "time": dates,
        "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
        "relative_humidity_2m": hourly.Variables(1).ValuesAsNumpy(),
        "precipitation": hourly.Variables(2).ValuesAsNumpy(),
        "pressure_msl": hourly.Variables(3).ValuesAsNumpy(),
        "surface_pressure": hourly.Variables(4).ValuesAsNumpy(),
        "cloud_cover": hourly.Variables(5).ValuesAsNumpy(),
        "wind_speed_10m": hourly.Variables(6).ValuesAsNumpy(),
    })

    df["latitude"] = latitude
    df["longitude"] = longitude
    df["location_id"] = f"{latitude}_{longitude}"
    return df


# Beispiel: Standorte abrufen (z.B. Städte mit Unwetterrisiko)
locations = [
    {"name": "Muenchen", "lat": 48.1351, "lon": 11.5820},
    {"name": "Hamburg", "lat": 53.5511, "lon": 9.9937},
]

raw_dfs = []
for loc in locations:
    df = fetch_weather_data(loc["lat"], loc["lon"])
    df["location_name"] = loc["name"]
    raw_dfs.append(df)

raw_df = pd.concat(raw_dfs, ignore_index=True)
print(f"✅ {len(raw_df)} Zeilen Rohdaten abgerufen")
print(raw_df.head())

✅ 384 Zeilen Rohdaten abgerufen
                       time  temperature_2m  relative_humidity_2m  \
0 2026-09-08 00:00:00+02:00       20.965000                  69.0   
1 2026-09-08 01:00:00+02:00       19.915001                  72.0   
2 2026-09-08 02:00:00+02:00       19.365000                  73.0   
3 2026-09-08 03:00:00+02:00       18.965000                  67.0   
4 2026-09-08 04:00:00+02:00       18.265001                  68.0   

   precipitation  pressure_msl  surface_pressure  cloud_cover  wind_speed_10m  \
0            0.0   1020.200012        960.624634          0.0        6.297428   
1            0.0   1020.200012        960.418518          0.0        5.692099   
2            0.0   1020.099976        960.216064          0.0        5.154415   
3            0.0   1019.400024        959.478149          0.0        5.991594   
4            0.0   1019.099976        959.057068          0.0        5.315336   

   latitude  longitude     location_id location_name  
0   48.1351

In [12]:
# Features engineeren (Rolling Windows, Aggregationen)
def engineer_features(df):
    """
    Erstellt Features, die für Unwetter-Prognosen entscheidend sind:
    - Rolling Windows für Trends
    - Druckabfall (Vorbote für Stürme)
    - Windböen-Anomalien
    """
    df = df.sort_values(["location_id", "time"]).copy()
    df["pressure_msl"] = df.get("pressure_msl", df.get("surface_pressure", pd.Series(0.0, index=df.index)))
    df["wind_gusts_10m"] = df.get("wind_gusts_10m", df.get("wind_speed_10m", pd.Series(0.0, index=df.index)))
    df["cape"] = pd.to_numeric(
        df.get("cape", pd.Series(0.0, index=df.index)),
        errors="coerce",
    ).fillna(0.0)

    grouped = df.groupby("location_id")

    # 🌀 Rolling Windows (z.B. 3h, 6h, 12h)
    for window in [3, 6, 12]:
        df[f"precip_rolling_sum_{window}h"] = grouped["precipitation"].transform(
            lambda x: x.rolling(window, min_periods=1).sum()
        )
        df[f"wind_gust_max_{window}h"] = grouped["wind_gusts_10m"].transform(
            lambda x: x.rolling(window, min_periods=1).max()
        )
        df[f"pressure_mean_{window}h"] = grouped["pressure_msl"].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )

    # Druckabfall als Sturm-Indikator (kritisch!)
    df["pressure_change_3h"] = grouped["pressure_msl"].transform(
        lambda x: x.diff(periods=3)
    )
    df["pressure_drop_rate"] = df["pressure_change_3h"] / 3

    # Windböen-Anomalie (Abweichung vom Mittelwert)
    df["wind_gust_anomaly"] = grouped["wind_gusts_10m"].transform(
        lambda x: x - x.rolling(24, min_periods=1).mean()
    )

    # Temperaturdifferenz (Fronten-Indikator)
    df["temp_change_3h"] = grouped["temperature_2m"].transform(
        lambda x: x.diff(periods=3)
    )

    # CAPE-Kategorien (Gewitterpotential)
    df["cape_risk_level"] = pd.cut(
        df["cape"], bins=[0, 500, 1500, 3000, float("inf")],
        labels=["niedrig", "moderat", "hoch", "extrem"]
    )

    # Label erstellen: Unwetter-Trigger (Beispiel-Logik)
    df["is_severe_weather"] = (
        (df["wind_gusts_10m"] > 60) |
        (df["precip_rolling_sum_3h"] > 20) |
        (df["pressure_drop_rate"] < -2)
    ).astype(int)

    return df

feature_df = engineer_features(raw_df)
print(feature_df.head())
print(f"✅ Features erstellt: {feature_df.shape[1]} Spalten")

                       time  temperature_2m  relative_humidity_2m  \
0 2026-09-08 00:00:00+02:00       20.965000                  69.0   
1 2026-09-08 01:00:00+02:00       19.915001                  72.0   
2 2026-09-08 02:00:00+02:00       19.365000                  73.0   
3 2026-09-08 03:00:00+02:00       18.965000                  67.0   
4 2026-09-08 04:00:00+02:00       18.265001                  68.0   

   precipitation  pressure_msl  surface_pressure  cloud_cover  wind_speed_10m  \
0            0.0   1020.200012        960.624634          0.0        6.297428   
1            0.0   1020.200012        960.418518          0.0        5.692099   
2            0.0   1020.099976        960.216064          0.0        5.154415   
3            0.0   1019.400024        959.478149          0.0        5.991594   
4            0.0   1019.099976        959.057068          0.0        5.315336   

   latitude  longitude  ... pressure_mean_6h precip_rolling_sum_12h  \
0   48.1351     11.582  ...

In [13]:
# Dataframe mit Features und Label erstellen
def build_final_dataframe(df):
    """
    Selektiert relevante Spalten und bereitet Primary Key + Event Time vor.
    """
    desired_columns = [
        "location_id",
        "location_name",
        "latitude",
        "longitude",
        "time",
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "pressure_msl",
        "surface_pressure",
        "cloud_cover",
        "wind_speed_10m",
        "wind_gusts_10m",
        "cape",
        "precip_rolling_sum_3h",
        "precip_rolling_sum_6h",
        "precip_rolling_sum_12h",
        "wind_gust_max_3h",
        "wind_gust_max_6h",
        "wind_gust_max_12h",
        "pressure_mean_3h",
        "pressure_mean_6h",
        "pressure_mean_12h",
        "pressure_change_3h",
        "pressure_drop_rate",
        "wind_gust_anomaly",
        "temp_change_3h",
        "is_severe_weather",
    ]

    feature_columns = [col for col in desired_columns if col in df.columns]
    final_df = df[feature_columns].copy()

    # Primary Key als eindeutige ID
    final_df["event_id"] = (
        final_df["location_id"] + "_" +
        final_df["time"].dt.strftime("%Y%m%d%H%M")
    )

    # Event Time sicherstellen
    final_df["event_time"] = final_df["time"]

    # NaN-Werte behandeln (z.B. am Anfang der Rolling Windows)
    final_df = final_df.dropna(subset=["pressure_drop_rate", "wind_gust_anomaly"]).copy()

    return final_df

final_df = build_final_dataframe(feature_df)
print(f"✅ Finaler DataFrame: {final_df.shape}")
print(final_df.head())

✅ Finaler DataFrame: (378, 30)
      location_id location_name  latitude  longitude  \
3  48.1351_11.582      Muenchen   48.1351     11.582   
4  48.1351_11.582      Muenchen   48.1351     11.582   
5  48.1351_11.582      Muenchen   48.1351     11.582   
6  48.1351_11.582      Muenchen   48.1351     11.582   
7  48.1351_11.582      Muenchen   48.1351     11.582   

                       time  temperature_2m  relative_humidity_2m  \
3 2026-09-08 03:00:00+02:00       18.965000                  67.0   
4 2026-09-08 04:00:00+02:00       18.265001                  68.0   
5 2026-09-08 05:00:00+02:00       17.765001                  75.0   
6 2026-09-08 06:00:00+02:00       18.065001                  72.0   
7 2026-09-08 07:00:00+02:00       18.115000                  70.0   

   precipitation  pressure_msl  surface_pressure  ...  pressure_mean_3h  \
3            0.0   1019.400024        959.478149  ...       1019.900004   
4            0.0   1019.099976        959.057068  ...       1019.53

In [14]:
# Feature Group im Hopsworks Feature Store erstellen und Daten einfügen

# Access the feature store
fs = project.get_feature_store()

# Feature Group
feature_group = fs.get_or_create_feature_group(
    name="weather_features_batch",
    version=1,
    primary_key=["event_id"],
    event_time="event_time",
    time_travel_format="HUDI",
    stream=True,
    description="Stündliche Wetterfeatures für die Unwetterprognose",
)

feature_group.stream = True
feature_group.insert(final_df, write_options={"wait_for_job": True})
print(f"✅ {len(final_df)} Zeilen in weather_features_batch geschrieben")

Uploading Dataframe: 100.00% |██████████| Rows 378/378 | Elapsed Time: 00:00 | Remaining Time: 00:00


Launching job: weather_features_batch_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/44167/jobs/named/weather_features_batch_1_offline_fg_materialization/executions


2026-09-15 09:19:24,891 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-09-15 09:19:31,224 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-09-15 09:23:01,836 INFO: Waiting for execution to finish. Current state: FINISHED. Final status: SUCCEEDED
2026-09-15 09:23:02,305 INFO: Waiting for log aggregation to finish.
2026-09-15 09:23:02,305 INFO: Execution finished successfully.


✅ 378 Zeilen in weather_features_batch geschrieben


In [15]:
# Feature Pipeline
def run_feature_pipeline(locations, fs=None, past_days=7, forecast_days=3, upload=False):
    """Erstellt Wetterfeatures und optional einen Hopsworks-Upload."""
    raw_dfs = []
    for loc in locations:
        df = fetch_weather_data(loc["lat"], loc["lon"], past_days, forecast_days)
        df["location_name"] = loc["name"]
        raw_dfs.append(df)

    raw_df = pd.concat(raw_dfs, ignore_index=True)
    feature_df = engineer_features(raw_df)
    final_df = build_final_dataframe(feature_df)

    weather_fg = None
    if upload:
        if fs is None:
            raise ValueError("Für upload=True muss ein Hopsworks Feature Store übergeben werden.")

        weather_fg = fs.get_or_create_feature_group(
            name="weather_features_batch",
            version=1,
            primary_key=["event_id"],
            event_time="event_time",
            time_travel_format="HUDI",
            stream=True,
            description="Stündliche Wetterfeatures für die Unwetterprognose",
        )
        weather_fg.stream = True
        weather_fg.insert(final_df, write_options={"wait_for_job": True})
        print(f"✅ {len(final_df)} Zeilen in weather_features_batch geschrieben")

    return final_df, weather_fg

# Die Feature Group wurde bereits in der vorherigen Zelle befüllt.
final_df, weather_fg = run_feature_pipeline(locations, fs, upload=False)
print(f"✅ End-to-End-Pipeline erfolgreich: {final_df.shape}")

✅ End-to-End-Pipeline erfolgreich: (474, 30)
